# Phase 6d Wave 2: Chiral SSB and the Gell-Mann–Oakes–Renner Relation — Technical Notebook

Companion to `papers/paper37_chiral_ssb/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/ChiralSSB_QCD.lean` (10 substantive theorems / 0 sorry / 0 new axioms; verified `propext, Classical.choice, Quot.sound` only via `lean_verify`).

**Structure (mirrors paper §1–§7):**
1. Quark condensate as order parameter: `QuarkCondensate` structure with $\sigma < 0$ invariant
2. FLAG-2021 lattice witness $\langle \bar q q \rangle = -0.0227$ GeV³
3. The GMOR relation $m_\pi^2 f_\pi^2 = -2 m_q \langle \bar q q \rangle$
4. PDG/FLAG numerical agreement to ~$2.5 \times 10^{-4}$ relative residual at the project's PDG-rounded working values (m_π = 0.137, f_π = 0.092 GeV)
5. Chiral-unbroken-phase contrapositive `chiral_unbroken_violates_gmor`
6. Tetrad-VEV / quark-condensate naturalness correctness-push
7. Cross-bridge to WetterichNJL scalar-channel upper bound
8. Figure: GMOR LHS/RHS bars + log-residual sweep
9. Lean theorem inventory

All numerical content imports from `src.chiral_ssb` — no inline physics redefinition.

## 1. Quark condensate as order parameter

Lean structure `QuarkCondensate { sigma : ℝ; sigma_neg : sigma < 0 }`. The negativity is a *structural invariant* — no `QuarkCondensate` object can be constructed with $\sigma \geq 0$. This encodes the order-parameter sign convention for $\langle \bar q q \rangle$.

FLAG-2021 lattice average: $\langle \bar q q \rangle \approx -(283 \text{ MeV})^3 \approx -0.0227 \text{ GeV}^3$.

In [ ]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.chiral_ssb import (
    QuarkCondensate, FLAG_LATTICE_VALUE,
    is_quark_condensate_candidate,
    PDG_M_PI, PDG_F_PI, PDG_M_Q,
    gmor_lhs, gmor_rhs, gmor_holds, gmor_pdg_match,
    H_TetradQuarkScalesNatural,
)
from src.core.visualizations import COLORS, fig_gmor_relation_verification
import math

sigma_scale_mev = abs(FLAG_LATTICE_VALUE.sigma) ** (1/3.0) * 1000
print(f'  FLAG-2021 lattice value σ = {FLAG_LATTICE_VALUE.sigma} GeV³  ({sigma_scale_mev:.1f} MeV cubed scale)')
print(f'  Negativity invariant       sigma < 0:   {FLAG_LATTICE_VALUE.sigma < 0}')

try:
    bad = QuarkCondensate(sigma=0.01)
except ValueError as e:
    print(f'  Positive σ rejected:       {e}')

## 2. PDG / FLAG numerical anchors

All four GMOR inputs come from independent measurements. There is no fitted parameter.

In [ ]:
print(f'  Project PDG-rounded m_π    = {PDG_M_PI} GeV   ({PDG_M_PI*1000:.1f} MeV; PDG charged π is 139.57 MeV, neutral 134.98)')
print(f'  Project PDG-rounded f_π    = {PDG_F_PI} GeV   ({PDG_F_PI*1000:.1f} MeV; PDG f_π is 92.4 MeV)')
print(f'  Project PDG-rounded m_q    = {PDG_M_Q} GeV   ({PDG_M_Q*1000:.2f} MeV, ≈ (m_u + m_d)/2 MS-bar at 2 GeV)')
print(f'  FLAG-2021 condensate    ⟨q̄q⟩  = {FLAG_LATTICE_VALUE.sigma} GeV³  (≈ -(283 MeV)³)')

## 3. The GMOR relation

$$m_\pi^2 \, f_\pi^2 \;=\; -2 \, m_q \, \langle \bar q q \rangle.$$

This is an algebraic consequence of soft-pion theorems and PCAC applied to QCD. **It is leading-order in chiral perturbation theory.** Higher-order chiPT corrections at the $\sim 1\%$ level are known to exist (NLO and beyond); the $\sim 0.01\%$ relative match reported here at the PDG-rounded central values sits *within* the NLO band, so it is consistent rather than tighter than chiPT precision. Compute LHS and RHS at PDG/FLAG central values and check agreement.

In [ ]:
lhs = gmor_lhs(PDG_M_PI, PDG_F_PI)
rhs = gmor_rhs(PDG_M_Q, FLAG_LATTICE_VALUE)
diff = abs(lhs - rhs)
rel_diff = diff / lhs
print(f'  LHS = m_π² f_π²            = {lhs:.6e} GeV⁴')
print(f'  RHS = -2 m_q ⟨q̄q⟩          = {rhs:.6e} GeV⁴')
print(f'  |LHS − RHS|                = {diff:.6e} GeV⁴')
print(f'  relative residual          = {rel_diff:.2e}    (~1 part in {1/rel_diff:.0f})')
print(f'  GMOR holds (tol 1e-4 GeV⁴): {gmor_holds(PDG_M_PI, PDG_F_PI, PDG_M_Q, FLAG_LATTICE_VALUE)}')
print(f'  Lean theorem gmor_pdg_match: |LHS - RHS| < 1e-4 GeV⁴ (proved by norm_num).')

## 4. Chiral-unbroken phase contrapositive

Lean theorem `chiral_unbroken_violates_gmor`: if $m_q > 0$, $m_\pi \neq 0$, $f_\pi \neq 0$, and $\sigma \geq 0$ (chiral-unbroken hypothesis, parametric over raw $\sigma$), then GMOR cannot hold — contradiction. The proof uses ALL four hypotheses (and was caught at first-pass review when an earlier version trivially closed via `qc.sigma_neg`, ignoring the GMOR equation).

**Numerical demonstration:** at $\sigma = 0$, RHS = 0, but LHS = $m_\pi^2 f_\pi^2 > 0$ → contradiction.

In [4]:
lhs_pdg = gmor_lhs(PDG_M_PI, PDG_F_PI)
for sigma_unbroken in [0.0, 0.01, 0.0227]:
    rhs_unbroken = -2.0 * PDG_M_Q * sigma_unbroken
    print(f'  σ = {sigma_unbroken:7.4f} GeV³ (unbroken or wrong sign)')
    print(f'    LHS (always > 0)        = {lhs_pdg:.6e} GeV⁴')
    print(f'    RHS (-2 m_q σ)           = {rhs_unbroken:.6e} GeV⁴')
    print(f'    GMOR holds?              = {abs(lhs_pdg - rhs_unbroken) < 1e-4}')
    print()
print('  → For positive m_q and non-zero pion sector, σ ≥ 0 forces RHS ≤ 0,')
print('    incompatible with LHS > 0. This is the contrapositive: chiral')
print('    unbroken phase (σ ≥ 0) violates GMOR.')

  σ =  0.0000 GeV³ (unbroken or wrong sign)
    LHS (always > 0)        = 1.588608e-04 GeV⁴
    RHS (-2 m_q σ)           = -0.000000e+00 GeV⁴
    GMOR holds?              = False

  σ =  0.0100 GeV³ (unbroken or wrong sign)
    LHS (always > 0)        = 1.588608e-04 GeV⁴
    RHS (-2 m_q σ)           = -7.000000e-05 GeV⁴
    GMOR holds?              = False

  σ =  0.0227 GeV³ (unbroken or wrong sign)
    LHS (always > 0)        = 1.588608e-04 GeV⁴
    RHS (-2 m_q σ)           = -1.589000e-04 GeV⁴
    GMOR holds?              = False

  → For positive m_q and non-zero pion sector, σ ≥ 0 forces RHS ≤ 0,
    incompatible with LHS > 0. This is the contrapositive: chiral
    unbroken phase (σ ≥ 0) violates GMOR.


## 5. Tetrad-VEV / quark-condensate naturalness correctness-push

Lean tracked-Prop `H_TetradQuarkScalesNatural`: the project's tetrad VEV scale $v_{\rm tetrad}$ and the quark-condensate scale $|\sigma|^{1/3}$ should agree to within an order of magnitude (10× either way). **Three-conjunct** structural Prop: (a) `0 < sigma_scale`, (b) `sigma_scale / 10 ≤ v_tetrad`, (c) `v_tetrad ≤ 10 * sigma_scale` — drop-conjunct test passes per conjunct, all three load-bearing (positivity of the $\sigma$-scale rules out the trivial-$\sigma$ degenerate point).

- Witness: unit ratio $v_{\rm tetrad} = |\sigma|^{1/3}$ satisfies the predicate.
- Falsifier 1: $v_{\rm tetrad}$ 100× too large (upper conjunct fails).
- Falsifier 2: $v_{\rm tetrad}$ 100× too small (lower conjunct fails).

In [5]:
sigma_scale = abs(FLAG_LATTICE_VALUE.sigma) ** (1/3.0)
for label, v_tetrad in [
    ('unit ratio (witness)',          sigma_scale),
    ('within-window: 5× larger',      5 * sigma_scale),
    ('within-window: 5× smaller',     sigma_scale / 5),
    ('falsifier: 100× larger',        100 * sigma_scale),
    ('falsifier: 100× smaller',       sigma_scale / 100),
]:
    h = H_TetradQuarkScalesNatural(v_tetrad=v_tetrad, sigma_scale=sigma_scale)
    print(f'  {label:35s}  v_tetrad = {v_tetrad:.4f} GeV   '
          f'lower={h.lower_bound}  upper={h.upper_bound}  holds={h.holds}')

  unit ratio (witness)                 v_tetrad = 0.2831 GeV   lower=True  upper=True  holds=True
  within-window: 5× larger             v_tetrad = 1.4157 GeV   lower=True  upper=True  holds=True
  within-window: 5× smaller            v_tetrad = 0.0566 GeV   lower=True  upper=True  holds=True
  falsifier: 100× larger               v_tetrad = 28.3145 GeV   lower=True  upper=False  holds=False
  falsifier: 100× smaller              v_tetrad = 0.0028 GeV   lower=False  upper=True  holds=False


## 6. Cross-bridge to WetterichNJL scalar-channel bound

Lean theorem `njl_scalar_bounded_consistent_with_chiral_broken` consumes BOTH the upstream `WetterichNJL.njl_scalar_upper_bound` (proved via `nlinarith` on `mul_le_mul_of_nonneg_left` with occupation-bound hypotheses $n_x, n_y \leq N$) AND the W2-internal `gmor_rhs_pos_of_quark_mass_pos` (uses `qc.sigma_neg`).

This is a *substantive* cross-bridge — the proof body uses both upstream calls; the conjunction is not decorative.

In [6]:
rhs_pos = gmor_rhs(PDG_M_Q, FLAG_LATTICE_VALUE)
print(f'  RHS at PDG: gmor_rhs > 0 (uses both PDG_M_Q > 0 and FLAG.sigma < 0):')
print(f'    {rhs_pos:.6e} GeV⁴ > 0:   {rhs_pos > 0}')
print()
print('  WetterichNJL.njl_scalar_upper_bound (Phase 5z.1):')
print('    For 0 ≤ g, occupation bounds 0 ≤ n_x ≤ N, 0 ≤ n_y ≤ N:')
print('      g · (n_x / N) · (n_y / N) ≤ g')
print('    The scalar channel is bounded above by the bare coupling.')
print()
print('  Cross-bridge `njl_scalar_bounded_consistent_with_chiral_broken`:')
print('    proves both (NJL bound) ∧ (GMOR RHS positivity at m_q > 0)')
print('    — ties Phase 5z.1 NJL infrastructure to the QCD chiral order parameter.')

  RHS at PDG: gmor_rhs > 0 (uses both PDG_M_Q > 0 and FLAG.sigma < 0):
    1.589000e-04 GeV⁴ > 0:   True

  WetterichNJL.njl_scalar_upper_bound (Phase 5z.1):
    For 0 ≤ g, occupation bounds 0 ≤ n_x ≤ N, 0 ≤ n_y ≤ N:
      g · (n_x / N) · (n_y / N) ≤ g
    The scalar channel is bounded above by the bare coupling.

  Cross-bridge `njl_scalar_bounded_consistent_with_chiral_broken`:
    proves both (NJL bound) ∧ (GMOR RHS positivity at m_q > 0)
    — ties Phase 5z.1 NJL infrastructure to the QCD chiral order parameter.


## 7. Figure — GMOR side-by-side bars + log-residual sweep

**Left panel.** Two bars, indistinguishable at this scale: $m_\pi^2 f_\pi^2$ vs $-2 m_q \langle \bar q q \rangle$, both $\approx 1.589 \times 10^{-4}$ GeV⁴. The annotation reports the residual $\approx 4 \times 10^{-8}$ GeV⁴ (relative residual ~$2.5 \times 10^{-4}$).

**Right panel.** The relative residual $|{\rm LHS} - {\rm RHS}(m_q)| / {\rm LHS}$ as a function of $m_q$ on a log scale. The vertical reference at PDG-rounded $m_q = 3.5$ MeV marks the GMOR-minimum point; the curve dips sharply at this point, confirming consistency.

In [7]:
# viz-ref: fig_gmor_relation_verification
fig_gmor_relation_verification().show()

## 8. Lean theorem inventory (10 substantive)

All theorems in `ChiralSSB_QCD.lean` are clean (zero `sorry`, zero new axioms; closure $\{\texttt{propext, Classical.choice, Quot.sound}\}$).

**Structures and definitions:**
- `QuarkCondensate { sigma : ℝ; sigma_neg : sigma < 0 }` — order parameter with negativity invariant.
- `flagLatticeValue : QuarkCondensate := ⟨-0.0227, by norm_num⟩` — FLAG-2021 witness.
- `IsQuarkCondensateCandidate` — predicate: $\sigma$ within fractional tolerance of observed.
- `gmor_lhs := m_\pi^2 f_\pi^2`, `gmor_rhs := -2 m_q \sigma`.

**Substantive theorems:**
1. `not_isQuarkCondensateCandidate_of_too_negative` — falsifier when $\sigma$ exceeds tolerance.
2. `gmor_lhs_nonneg` — LHS $\geq 0$ (product of squares).
3. `gmor_lhs_pos_iff` — LHS $> 0$ iff both pion-sector quantities are non-zero.
4. `gmor_rhs_pos_of_quark_mass_pos` — load-bearing; uses both $m_q > 0$ and $\sigma < 0$.
5. `gmor_pdg_match` — $|{\rm LHS} - {\rm RHS}| < 10^{-4}$ at PDG/FLAG values, by `norm_num`.
6. `chiral_unbroken_violates_gmor` — contrapositive parametric over raw $\sigma \geq 0$.
7. `H_TetradQuarkScalesNatural` (tracked Prop) — naturalness correctness-push.
8. `H_TetradQuarkScalesNatural_witness` — unit-ratio existence.
9. `H_TetradQuarkScalesNatural_falsifier_super_large` / `_super_small` — two falsifiers.
10. `njl_scalar_bounded_consistent_with_chiral_broken` — cross-bridge to `WetterichNJL.njl_scalar_upper_bound`.

Discipline trend: 6c.3 = 12, 6b.1 = 5, 6d.1 = 6, **6d.2 = 4** (1 first-pass + 2 second-pass + 1 third-pass). Multi-pass review until two consecutive clean passes; first-pass discipline + ruthless review.